In this assignment, you will implement Triton-based implementation of a matrix multiplication + ReLU + add kernel. The kernel computes the matrix function **D = ReLU(A × B + C)** where **A** is of shape *(M, K)*, **B** is of shape *(K, N)*, and **C & D** are of shape *(M, N)*. We will break the kernel down into four main steps:

1. **Tile Assignment**  
2. **Shared Memory Tiling + Cooperative Fetching**  
3. **Register Tiling (Accumulator)**  
4. **Operator Fusion**
5. **Write Cache/Epilogue (Store Results)**

### Setup
#### **VERY IMPORTANT:**
Please use Google Colab's free T4 GPU to run the notebook. You will be graded based on your performance and speedup when running on the T4 GPU. Different GPUs have different accelerations based on floating point precision, tensor cores, etc, so please use the Colab GPU when coding and evaluating your performance.

We also understand that there will be some issues in this assignment regarding precision and performance on fp32 tensors. Since T4's have known issues with fp32 precision (https://github.com/NVIDIA/TensorRT-LLM/issues/396#issuecomment-1817806895) and their tensor cores do not have compatibility with fp32 acceleration, your speedup on matmul may not be as good as you expect, if you run on newer generations of GPUs. However, please still use T4 GPU on Colab to complete this assignment as we have calculated our benchmarks, accuracy tolerance, and implementation based on Colab. We want you to be able to finish this assignment for free and not have to pay for a GPU.

**Note:** Since T4s do not support fp32 precision well, please ALWAYS do matmul (and accumulation) in fp16 precision for the purposes of this assignment. Changing it will lead to different performance metrics or broken code, which you do not want.

In [1]:
import torch
import triton
import triton.language as tl
import time

In [2]:
def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"

In [3]:
def is_hip_mi200():
    target = triton.runtime.driver.active.get_current_target()
    return target.backend == 'hip' and target.arch == 'gfx90a'

### Tile Assignment
**Tile Assignment** is the process by which the overall matrix **C** is divided into smaller submatrices (tiles). Each kernel instance (a GPU “program” or thread block) is responsible for computing one tile of **C**. This division allows the computation to be parallelized across many threads.

Each kernel launch instance (denoted by a unique program id `pid`) should be mapped to a specific tile in **C**. The tile is defined by two indices: one for the row (denoted `pid_m`) and one for the column (`pid_n`).

### Shared Memory Tiling + Cooperative Fetching
**Shared Memory Tiling** is used to load sub-tiles (smaller blocks) of matrices A and B into faster, on-chip memory. Threads within a kernel instance load these sub-tiles, reducing the number of global memory accesses.

Some things to keep in mind:
- You may need to use `tl.arange` to help compute offsets for the rows and columns.
- Use masks to make sure that out-of-bound memory accesses do not occur.

### Register Tiling (Accumulator)
**Register Tiling** is the use of registers to hold intermediate results (partial sums) during the matrix multiplication. For this section, you will need to use an accumulator (a BLOCK_SIZE_M by BLOCK_SIZE_N matrix initialized to zeros) to accumulate results of dot products computed over tiles.

As mentioned, use FP16 for the accumulator, when developing and testing on Colab with T4 GPU.

### Add and ReLU Fusion
In this step, we fuse the element-wise addition of matrix C and the final ReLU activation directly into the matmul kernel to optimize performance by reducing memory traffic and kernel launch overhead. After computing the matrix multiplication and storing the results in the accumulator, you must load the corresponding tile from matrix C and add it element-wise to the activated accumulator. Then apply the ReLU function using an element-wise maximum operation to set all negative values to zero.

This fusion of operations avoids writing intermediate results back to global memory and then reloading them for further processing, minimizing latency and making more efficient use of the GPU’s memory hierarchy.

### Write Cache/Epilogue
In this step, we write the tile of C back to global memory. This final step ensures that the computed results are stored in the correct locations in the output matrix C. Be sure to also use a mask to prevent invalid indices from being written to. (Use `tl.store` to store your tiles.)

In [4]:
import triton
import triton.language as tl

BLOCK_M = 128
BLOCK_N = 128
BLOCK_K = 16
GROUP_SIZE_M = 4
UNROLL = 4

@triton.jit
def matmul_add_relu_kernel_fp16(
    a_ptr, b_ptr, c_ptr, d_ptr,
    M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
    stride_am: tl.constexpr, stride_ak: tl.constexpr,
    stride_bk: tl.constexpr, stride_bn: tl.constexpr,
    stride_cm: tl.constexpr, stride_cn: tl.constexpr,
    stride_dm: tl.constexpr, stride_dn: tl.constexpr,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
    UNROLL: tl.constexpr,
):
    # -------------------------
    # Tile assignment (grouped)
    # -------------------------
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)

    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = tl.minimum(num_pid_m - first_pid_m, GROUP_SIZE_M)

    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    m_mask = offs_m < M
    n_mask = offs_n < N
    out_mask = m_mask[:, None] & n_mask[None, :]

    # FP16 accumulator (requirement)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float16)

    k_tiles = tl.cdiv(K, BLOCK_K)

    # ---- SAFE compiler hints (1D only) ----
    # These are the ones that won't crash your Triton version.
    tl.multiple_of(offs_n, 8)   # BN is multiple of 8 in good configs
    tl.multiple_of(offs_k, 8)   # BK 16/32 -> good
    tl.max_contiguous(offs_n, BLOCK_N)
    tl.max_contiguous(offs_k, BLOCK_K)

    # Base pointers (k=0)
    a_ptrs0 = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs0 = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    k = 0
    while k < k_tiles:
        for u in tl.static_range(0, UNROLL):
            kk = k + u
            if kk < k_tiles:
                k_offs = kk * BLOCK_K + offs_k
                k_mask = k_offs < K

                a_ptrs = a_ptrs0 + kk * BLOCK_K * stride_ak
                b_ptrs = b_ptrs0 + kk * BLOCK_K * stride_bk

                a = tl.load(a_ptrs, mask=m_mask[:, None] & k_mask[None, :], other=0.0)
                b = tl.load(b_ptrs, mask=k_mask[:, None] & n_mask[None, :], other=0.0)

                # Keep the experiment: force fp16 dot output
                acc += tl.dot(a, b, out_dtype=tl.float16)

        k += UNROLL

    # Fuse add + ReLU
    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    c_tile = tl.load(c_ptrs, mask=out_mask, other=0.0)

    out = tl.maximum(acc + c_tile, 0.0)

    d_ptrs = d_ptr + offs_m[:, None] * stride_dm + offs_n[None, :] * stride_dn
    tl.store(d_ptrs, out, mask=out_mask)


In [5]:
def matmul_add_relu_fp16(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    assert c.shape == (M, N)

    d = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N),)

    matmul_add_relu_kernel_fp16[grid](
        a, b, c, d,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        d.stride(0), d.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        GROUP_SIZE_M=GROUP_SIZE_M,
        UNROLL=UNROLL,
        num_warps=8,
        num_stages=3,
    )

    return d


In [6]:
# Reference implementation using PyTorch
def reference_matmul_add_relu(A, B, C):
    result = torch.matmul(A, B).add(C).relu_()
    return result

In [7]:
# -----------------------------------------------------------------------------
# Accuracy Tests
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    torch.manual_seed(59)
    a = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    b = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    c = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    triton_output = matmul_add_relu_fp16(a, b, c)
    torch_output = reference_matmul_add_relu(a, b, c)
    print(f"triton_output_with_fp16_inputs={triton_output}")
    print(f"torch_output_with_fp16_inputs={torch_output}")
    rtol = 0.1 if is_hip_mi200() else 0.08
    if torch.allclose(triton_output, torch_output, atol=1.0, rtol=rtol):
        diff = triton_output - torch_output
        abs_diff = torch.abs(diff)
        max_abs_diff = torch.max(abs_diff)
        print("✅ Triton and Torch match, max_abs_diff=", max_abs_diff)
    else:
        diff = triton_output - torch_output
        abs_diff = torch.abs(diff)
        max_abs_diff = torch.max(abs_diff)
        print(f"❌ Triton and Torch differ: {max_abs_diff=}")

triton_output_with_fp16_inputs=tensor([[49.6250,  0.0000,  0.0000,  ...,  0.0000,  4.2656,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000, 10.6406, 19.4531],
        [17.7812,  0.0000,  0.0000,  ...,  0.0000, 26.8125,  0.0000],
        ...,
        [19.2031,  0.0000, 14.6875,  ...,  0.0000, 29.2344,  0.0000],
        [21.6406,  6.1602,  0.0000,  ..., 11.8984,  2.7344,  0.0000],
        [ 0.0000,  9.2891,  8.3047,  ...,  0.0000, 26.5312,  0.0000]],
       device='cuda:0', dtype=torch.float16)
torch_output_with_fp16_inputs=tensor([[49.5312,  0.0000,  0.0000,  ...,  0.0000,  4.2812,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000, 10.6484, 19.4375],
        [17.7969,  0.0000,  0.0000,  ...,  0.0000, 26.8125,  0.0000],
        ...,
        [19.2031,  0.0000, 14.6797,  ...,  0.0000, 29.2344,  0.0000],
        [21.7656,  6.1211,  0.0000,  ..., 11.8438,  2.7207,  0.0000],
        [ 0.0000,  9.2422,  8.2344,  ...,  0.0000, 26.5625,  0.0000]],
       device='cuda:0', dt

### Grid Search
In this section, you will have to perform grid search or manually find values for the hyperparameter block sizes (`BLOCK_M`, `BLOCK_N`, `BLOCK_K`). We have provided parameters that should result in you achieving around 0.2x speedup, but you will have to search for different block sizes to achieve better speedup.

In [8]:
# -----------------------------------------------------------------------------
# Performance Benchmark
# IMPORTANT: DO NOT CHANGE THIS CODE.
# THIS IS THE EXACT CODE THAT WILL BE USED TO GRADE YOUR IMPLEMENTATION.
# ANY CHANGES TO THIS CODE (INCLUDING DIMENSIONS, REPEATS, etc.)
# WILL CAUSE YOU TO HAVE DIFFERENT SPEEDUP RESULTS.
# -----------------------------------------------------------------------------
M = 2048
K = 2048
N = 2048

# KEEP THESE MATRICES IN FP16. FP32 WILL NOT PROVIDE ACCURATE RESULTS
A = torch.randn((M, K), device="cuda", dtype=torch.float16)
B = torch.randn((K, N), device="cuda", dtype=torch.float16)
C = torch.randn((M, N), device="cuda", dtype=torch.float16)

# warmup
_ = matmul_add_relu_fp16(A, B, C)
_ = reference_matmul_add_relu(A, B, C)

REPEATS = 5000

# time your implementation
print("Triton implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = matmul_add_relu_fp16(A, B, C)
torch.cuda.synchronize()
triton_time = (time.perf_counter() - start) / REPEATS

# time pytorch
print("PyTorch implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = reference_matmul_add_relu(A, B, C)
torch.cuda.synchronize()
torch_time = (time.perf_counter() - start) / REPEATS

print(f"Performance comparison for matrix multiplication ({M}x{K} @ {K}x{N}):")
print(f"Triton implementation: {triton_time*1000:.2f} ms")
print(f"PyTorch implementation: {torch_time*1000:.2f} ms")

print(f"\nSpeedup of Triton vs PyTorch: {torch_time/triton_time:.2f}x")

Triton implementation
PyTorch implementation
Performance comparison for matrix multiplication (2048x2048 @ 2048x2048):
Triton implementation: 2.88 ms
PyTorch implementation: 1.00 ms

Speedup of Triton vs PyTorch: 0.35x


In [9]:
import time, statistics
import torch, triton

@torch.no_grad()
def run_config(A, B, C, BM, BN, BK, GSM, nw, ns, unroll, reps=25):
    M, K = A.shape
    _, N = B.shape
    D = torch.empty((M, N), device="cuda", dtype=torch.float16)
    grid = (triton.cdiv(M, BM) * triton.cdiv(N, BN),)

    # Warmup
    for _ in range(5):
        matmul_add_relu_kernel_fp16[grid](
            A, B, C, D, M, N, K,
            A.stride(0), A.stride(1),
            B.stride(0), B.stride(1),
            C.stride(0), C.stride(1),
            D.stride(0), D.stride(1),
            BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK,
            GROUP_SIZE_M=GSM,
            UNROLL=unroll,
            num_warps=nw, num_stages=ns,
        )
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(reps):
        matmul_add_relu_kernel_fp16[grid](
            A, B, C, D, M, N, K,
            A.stride(0), A.stride(1),
            B.stride(0), B.stride(1),
            C.stride(0), C.stride(1),
            D.stride(0), D.stride(1),
            BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK,
            GROUP_SIZE_M=GSM,
            UNROLL=unroll,
            num_warps=nw, num_stages=ns,
        )
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / reps


# Benchmark matrices (grading dims)
M = N = K = 2048
A = torch.randn((M, K), device="cuda", dtype=torch.float16)
B = torch.randn((K, N), device="cuda", dtype=torch.float16)
C = torch.randn((M, N), device="cuda", dtype=torch.float16)

print("== Targeted sweep v2: expand tiles + fp16 dot out_dtype ==")

tiles = [
    (64, 128, 16),
    (128, 64, 16),
    (128, 128, 16),
    (64, 128, 32),
    (128, 64, 32),
]

warps   = [4, 8]           # we keep 4 but expect it to be poison sometimes
stages  = [2, 3, 4, 5]     # enough to pipeline without exploding compile time
GSMs    = [4, 8, 16]       # no need to sweep 1..32 anymore
UNROLLS = [2, 4]           # UNROLL=8 is proven bad on your logs

results = []
idx = 0
total = len(tiles) * len(warps) * len(stages) * len(GSMs) * len(UNROLLS)

for BM, BN, BK in tiles:
    for nw in warps:
        for ns in stages:
            for GSM in GSMs:
                for unroll in UNROLLS:
                    idx += 1
                    try:
                        # quick probe first to avoid burning time on poison configs
                        t_probe = run_config(A, B, C, BM, BN, BK, GSM, nw, ns, unroll, reps=5)
                        ms_probe = t_probe * 1000
                        if ms_probe > 20.0:
                            print(f"[{idx:4d}/{total}] SKIP (slow) | "
                                  f"BM={BM} BN={BN} BK={BK} GSM={GSM} nw={nw} ns={ns} UNROLL={unroll} -> {ms_probe:.2f} ms")
                            continue

                        t = run_config(A, B, C, BM, BN, BK, GSM, nw, ns, unroll, reps=25)
                        ms = t * 1000
                        results.append((ms, BM, BN, BK, GSM, nw, ns, unroll))
                        print(f"[{idx:4d}/{total}] {ms:6.2f} ms | "
                              f"BM={BM:3d} BN={BN:3d} BK={BK:2d} GSM={GSM:2d} "
                              f"nw={nw} ns={ns} UNROLL={unroll}")
                    except Exception as e:
                        print(f"[{idx:4d}/{total}] SKIP | BM={BM} BN={BN} BK={BK} GSM={GSM} "
                              f"nw={nw} ns={ns} UNROLL={unroll} ({type(e).__name__})")

results.sort()
print("\nTop-10:")
for ms, BM, BN, BK, GSM, nw, ns, unroll in results[:10]:
    print(f"  {ms:.2f}ms | BM={BM} BN={BN} BK={BK} GSM={GSM} nw={nw} ns={ns} UNROLL={unroll}")


== Targeted sweep v2: expand tiles + fp16 dot out_dtype ==
[   1/240]   3.02 ms | BM= 64 BN=128 BK=16 GSM= 4 nw=4 ns=2 UNROLL=2
[   2/240]   2.90 ms | BM= 64 BN=128 BK=16 GSM= 4 nw=4 ns=2 UNROLL=4
[   3/240]   2.96 ms | BM= 64 BN=128 BK=16 GSM= 8 nw=4 ns=2 UNROLL=2
[   4/240]   2.87 ms | BM= 64 BN=128 BK=16 GSM= 8 nw=4 ns=2 UNROLL=4
[   5/240]   3.05 ms | BM= 64 BN=128 BK=16 GSM=16 nw=4 ns=2 UNROLL=2
[   6/240]   2.95 ms | BM= 64 BN=128 BK=16 GSM=16 nw=4 ns=2 UNROLL=4
[   7/240]   3.02 ms | BM= 64 BN=128 BK=16 GSM= 4 nw=4 ns=3 UNROLL=2
[   8/240]   2.92 ms | BM= 64 BN=128 BK=16 GSM= 4 nw=4 ns=3 UNROLL=4
[   9/240]   2.96 ms | BM= 64 BN=128 BK=16 GSM= 8 nw=4 ns=3 UNROLL=2
[  10/240]   2.89 ms | BM= 64 BN=128 BK=16 GSM= 8 nw=4 ns=3 UNROLL=4
[  11/240]   3.04 ms | BM= 64 BN=128 BK=16 GSM=16 nw=4 ns=3 UNROLL=2
[  12/240]   2.91 ms | BM= 64 BN=128 BK=16 GSM=16 nw=4 ns=3 UNROLL=4
[  13/240]   3.04 ms | BM= 64 BN=128 BK=16 GSM= 4 nw=4 ns=4 UNROLL=2
[  14/240]   2.91 ms | BM= 64 BN=128 BK=16 G

### Grading
We will grade you on completion of each subpart, as well as overall performance. We provide a list of thresholds below for speedup and how many points you will get for each speedup multiplier with respect to the reference PyTorch implementation provided in `reference_matmul` in the notebook.

- <0.2x speedup: grade subject to your implementation
- 0.2 - 0.27x speedup: 60 / 100 points
- 0.27 - 0.32 x speedup: 80 / 100 points
- 0.32x speedup and above: 100 / 100 points

Please make sure to test on the GPUs that we have mentioned here. Speedup is much much different on different GPUs! **AND DO NOT CHANGE THE CODE** in the performance benchmark section of the notebook. We are only testing on 2048x2048 fp16 matmul + add + relu, with 5000 iterations.

For submission, **please run your jupyter notebook fully with your best submission speedup**. This way we can have an easier time grading and looking at your implementation!

Please upload `matmul_triton.ipynb` to gradescope. This assignment is not automatically graded, and you will not receive immediate feedback upon submission. You can submit multiple times, but only your final submission will be graded.